# 01 · Auditoría de fuga de datos (Fase 0)

**Objetivo:** confirmar con datos reales qué información se conoce **al convocar** un proceso,
antes de entrenar el modelo de monto esperado con datos reales.

Una feature tiene *fuga de datos* si en la práctica solo se conoce después de adjudicar: el
modelo "haría trampa" y sus métricas serían engañosas. La guardia del código
(`src/contrataciones/features/catalog.py`) ya rechaza las columnas del resultado; esta auditoría
revisa lo que el código no puede saber: **cuándo publica el OECE cada dato**.

**Entradas:** la capa gold (`data/curated`), las páginas crudas de la API (`data/raw/api`, si
corriste `contrataciones run --source api`) y, de forma opcional, la historia de versiones de la
API (requiere conexión).

**Con datos reales:**

```bash
contrataciones download --year 2025
contrataciones run --source bulk --year 2025          # capa gold real
contrataciones run --source api --month 2025-09       # páginas crudas para la sección 4
```

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"          # capa gold
FEATURES = ROOT / "data" / "features"        # dataset de modelado
PREDICTIONS = ROOT / "data" / "predictions"  # monto esperado y atípicos
MODELS = ROOT / "models"                     # modelos entrenados
RAW_API = ROOT / "data" / "raw" / "api"      # páginas crudas de la API del OECE
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
COLORES_DESVIACION = {
    "Normal": "#C8CFD8",
    "Sobrecosto inusual": "#B5533C",
    "Subcosto inusual": "#3E6A8A",
    "Revisar datos": "#C9A227",
}
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
porcentaje = mticker.PercentFormatter(1.0)
print("Proyecto:", ROOT)

# Historia de versiones en la API (sección 5): una consulta por proceso adjudicado.
CONSULTAR_API = False
LIMITE_CONSULTAS = 200

In [ ]:
from contrataciones.features import audit
from contrataciones.features.dataset import build_modeling_table, load_gold, read_manifest
from contrataciones.utils.config_loader import load_config

config = load_config("model_config")
manifest = read_manifest(CURATED)
gold = load_gold(CURATED)
fp = gold["fact_proceso"]
table = build_modeling_table(gold, config["population"])
fuente = manifest.get("source")
print(f"Corrida {manifest.get('run_id')} | fuente: {fuente} | procesos: {len(fp):,}")
if fuente == "sample":
    print("Datos SINTÉTICOS: sirven para probar el notebook, pero la auditoría no es concluyente.")

## 1. ¿Qué columnas candidatas vienen vacías?

Una feature casi siempre vacía no aporta información aunque se conozca al convocar.

In [ ]:
vacios = audit.missing_shares(fp)
vacios.style.format({"pct_vacios": "{:.1%}", "pct_vacios_adjudicados": "{:.1%}"})

## 2. Postores: ¿se conocen antes del resultado?

Si los postores aparecen solo en procesos con resultado (buena pro o desierto), el número de
postores es información **posterior** a la convocatoria. Además es una consecuencia del proceso:
incluirlo haría que el modelo "explique" el sobreprecio por poca competencia, que es justo lo
que se quiere detectar (y RF01 ya lo cubre).

In [ ]:
audit.bidders_by_outcome(fp)

## 3. Monto adjudicado igual al valor referencial

Un adjudicado idéntico al referencial al céntimo puede ser legítimo (convenios, algunas obras)
o una señal de que el valor referencial se publicó recién con la buena pro. Si en un
procedimiento es la norma, el valor referencial de ese procedimiento no es una feature
confiable "al convocar".

In [ ]:
igualdad = audit.equality_by_segment(table)
principales = igualdad.groupby("metodo_detalle")[["n_procesos", "n_igual_referencial"]].sum()
principales["pct_igual_referencial"] = principales["n_igual_referencial"] / principales["n_procesos"]
principales = principales.sort_values("n_procesos", ascending=False).head(10)
ax = principales["pct_igual_referencial"].sort_values().plot.barh(
    title="Adjudicado = referencial por procedimiento", color=PALETA[1])
ax.xaxis.set_major_formatter(porcentaje); ax.set_ylabel(""); plt.show()
principales.style.format({"pct_igual_referencial": "{:.1%}"})

## 4. Páginas crudas de la API: información protegida por ley

El OECE marca con `hasTenderInformationProtectedByLaw` los procesos cuya información está
protegida durante la selección. Si todos los protegidos están sin adjudicar y todos los
adjudicados aparecen sin protección, la marca se levanta con la buena pro: la versión compilada
(la que usa el pipeline) ya no permite saber qué era público al convocar.

In [ ]:
registros = list(audit.iter_raw_records(RAW_API)) if RAW_API.exists() else []
if registros:
    marcas = audit.raw_record_flags(registros)
    print(f"{len(marcas):,} procesos en {RAW_API}")
    display(pd.crosstab(
        [marcas["informacion_protegida"], marcas["tiene_valor_referencial"]],
        marcas["tiene_adjudicacion"],
    ))
    adjudicados = marcas[marcas["tiene_adjudicacion"]]
    print(f"Adjudicados con monto = referencial: {adjudicados['adjudicado_igual_referencial'].mean():.1%}")
else:
    print("No hay páginas crudas de la API. Ejecuta: contrataciones run --source api --month AAAA-MM")

## 5. Historia de versiones (opcional, requiere conexión)

Para cada proceso adjudicado se lee la última versión **anterior** a la buena pro y se compara su
valor referencial con el final. Hace una consulta por proceso (con pausas de cortesía), así que
está desactivada por defecto: cambia `CONSULTAR_API = True` en la primera celda.

| Diagnóstico | Lectura |
| --- | --- |
| Valor estable | El referencial ya era público antes de la buena pro: feature válida |
| Valor revelado con la buena pro | Era 0 o no estaba antes: no era público al convocar |
| Valor cambia | Se modificó durante el proceso: revisar |
| Sin versión previa | La historia de la API empieza ya adjudicada: no se puede verificar |

In [ ]:
if CONSULTAR_API and registros:
    cliente = audit.client_from_config(load_config("api_config")["oece_ocds"])
    historia = audit.release_history_check(registros, cliente, limit=LIMITE_CONSULTAS)
    display(historia["diagnostico"].value_counts())
    display(historia.groupby("metodo")["diagnostico"].value_counts().unstack(fill_value=0))
else:
    print("Consulta a la API desactivada (CONSULTAR_API = False) o sin páginas crudas.")

## Decisión

Evidencia preliminar (60 procesos reales de 2026-09, revisados a mano el 2026-10-20):

| Hallazgo | Evidencia | Decisión en el modelo |
| --- | --- | --- |
| Los postores se publican con el resultado | Ausentes en las versiones previas; 23 adjudicados y 3 desiertos los tienen | `n_postores` prohibida en el código |
| El valor referencial puede revelarse con la buena pro | 3 de 7 adjudicados verificables tenían valor 0 antes; 2 de ellos igual al adjudicado | `log_valor_referencial` es **condicional** |
| Muchos adjudicados iguales al referencial | 15 de 23 | Se informa por segmento y en el reporte |
| Plazo de presentación vacío | 100 % (inicio = fin) | `plazo_presentacion_dias` deshabilitada |

Con los resultados de esta auditoría sobre datos reales, decide y registra en
`config/model_config.yaml`:

* `population.exclude_methods` / `exclude_categories`: procedimientos o categorías donde el
  valor referencial no es público al convocar (por ejemplo, si la mayoría tiene diagnóstico
  *Valor revelado con la buena pro*).
* Si el problema es general, el siguiente paso es capturar el valor referencial de la primera
  versión publicada (cambio en la ingesta, a evaluar).

Después: `contrataciones-ml train` y revisa `notebooks/ml/03_modelos_y_baselines.ipynb`.